# โหลด CTSpine1K Dataset จาก Hugging Face

โหลดคู่ volume-label จาก https://huggingface.co/datasets/alexanderdann/CTSpine1K/tree/main/raw_data

- `raw_data/volumes/COLONOG/` → CT volumes
- `raw_data/labels/COLONOG/` → Segmentation masks

In [1]:
# !pip install huggingface_hub nibabel

# Order
## trainset: 
### COLONOG --> 1-480
### HN_xx (HNSCC) --> 481-500
### liver (MSD) ---> 501-590
### COVID ---> 591-610

In [12]:
from huggingface_hub import HfApi, hf_hub_download
from pathlib import Path

REPO_ID = "alexanderdann/CTSpine1K"
SPLIT_FILE = Path("data_split.txt")   # ถ้า notebook อยู่ในโฟลเดอร์ check
TARGET_SPLIT = "trainset"             # เปลี่ยนเป็น test_public หรือ test_private ได้
SELECT_RANKS = "1-305"               # เว้นว่าง "" = โหลดทั้ง split

OUT_DIR = Path("./dummy")
IMAGES_DIR = OUT_DIR / "images"
MASKS_DIR = OUT_DIR / "masks"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MASKS_DIR.mkdir(parents=True, exist_ok=True)

def parse_split_file(path):
    groups = {}
    current = None

    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line:
            continue
        if line.endswith(":"):
            current = line[:-1]
            groups[current] = []
        else:
            if current is not None:
                groups[current].append(line)

    return groups

def parse_rank_selection(rank_text, max_len):
    selected = set()
    if not rank_text or not rank_text.strip():
        return []

    for token in rank_text.split(","):
        token = token.strip()
        if not token:
            continue

        if "-" in token:
            a, b = token.split("-", 1)
            a, b = a.strip(), b.strip()
            if a.isdigit() and b.isdigit():
                start, end = int(a), int(b)
                if start > end:
                    start, end = end, start
                for r in range(start, end + 1):
                    if 1 <= r <= max_len:
                        selected.add(r)
        else:
            if token.isdigit():
                r = int(token)
                if 1 <= r <= max_len:
                    selected.add(r)

    return sorted(selected)

def normalize_id(name):
    base = Path(name).name
    if base.endswith(".nii.gz"):
        base = base[:-7]
    if base.endswith("_seg"):
        base = base[:-4]
    if base.endswith("_label"):
        base = base[:-6]
    return base

# 1) อ่าน split และเลือกตามอันดับ (ถ้าระบุ)
split_groups = parse_split_file(SPLIT_FILE)
all_names_in_split = split_groups.get(TARGET_SPLIT, [])
rank_list = parse_rank_selection(SELECT_RANKS, len(all_names_in_split))

if rank_list:
    wanted_names = [all_names_in_split[r - 1] for r in rank_list]
    print(f"เลือกอันดับ: {rank_list[:15]}{' ...' if len(rank_list) > 15 else ''}")
else:
    wanted_names = all_names_in_split
    print("ไม่ได้ระบุอันดับ -> ใช้ทั้งหมดใน split")

wanted_ids = {normalize_id(x) for x in wanted_names}
print(f"split={TARGET_SPLIT}, selected cases={len(wanted_ids)}")

# 2) list ไฟล์ใน HF
api = HfApi()
all_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")

volume_files = [
    f for f in all_files
    if f.startswith("raw_data/volumes/") and f.endswith(".nii.gz")
]
label_files = [
    f for f in all_files
    if f.startswith("raw_data/labels/") and f.endswith(".nii.gz")
]

# 3) index ตาม patient id
vol_map = {}
for vf in volume_files:
    pid = normalize_id(vf)
    vol_map[pid] = vf

lbl_map = {}
for lf in label_files:
    pid = normalize_id(lf)
    lbl_map[pid] = lf

# 4) สร้างคู่ที่อยู่ใน split และมีทั้ง image+mask
pairs = []
missing = []
for pid in sorted(wanted_ids):
    v = vol_map.get(pid)
    l = lbl_map.get(pid)
    if v and l:
        pairs.append((pid, v, l))
    else:
        missing.append(pid)

print(f"matched pairs={len(pairs)}")
print(f"missing in HF or no pair={len(missing)}")
if missing:
    print("ตัวอย่างที่หาไม่เจอ:", missing[:10])


# พรีวิวอันดับที่จะโหลดก่อนเริ่มดาวน์โหลดจริง
print(f"TARGET_SPLIT = {TARGET_SPLIT}")
print(f"SELECT_RANKS = {SELECT_RANKS if SELECT_RANKS else 'ALL'}")
print(f"จำนวน cases ใน split = {len(all_names_in_split)}")

if rank_list:
    print("อันดับที่เลือก:", rank_list)
    print("รายการที่จะโหลด:")
    for idx, name in enumerate(wanted_names, 1):
        print(f"  {idx}. {name}")
else:
    print("ไม่ได้ระบุ SELECT_RANKS -> จะโหลดทั้งหมดใน split")
    for idx, name in enumerate(wanted_names[:30], 1):
        print(f"  {idx}. {name}")
    if len(wanted_names) > 30:
        print(f"  ... และอีก {len(wanted_names) - 30} รายการ")

เลือกอันดับ: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] ...
split=trainset, selected cases=305
matched pairs=305
missing in HF or no pair=0
TARGET_SPLIT = trainset
SELECT_RANKS = 1-305
จำนวน cases ใน split = 610
อันดับที่เลือก: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 

In [13]:
# 5) download
downloaded = 0
skipped = 0
for i, (pid, vol_path, lbl_path) in enumerate(pairs, 1):
    img_dest = IMAGES_DIR / f"{pid}.nii.gz"
    msk_dest = MASKS_DIR / f"{pid}_seg.nii.gz"

    if img_dest.exists() and msk_dest.exists():
        skipped += 1
        continue

    hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=vol_path,
        local_dir=str(IMAGES_DIR),
        local_dir_use_symlinks=False
    )
    downloaded_img_name = Path(vol_path).name
    src_img = IMAGES_DIR / downloaded_img_name
    if src_img.exists() and src_img != img_dest:
        src_img.rename(img_dest)

    hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=lbl_path,
        local_dir=str(MASKS_DIR),
        local_dir_use_symlinks=False
    )
    downloaded_lbl_name = Path(lbl_path).name
    src_lbl = MASKS_DIR / downloaded_lbl_name
    if src_lbl.exists() and src_lbl != msk_dest:
        src_lbl.rename(msk_dest)

    downloaded += 1
    if i % 20 == 0:
        print(f"progress {i}/{len(pairs)}")

print("done")
print("downloaded =", downloaded)
print("skipped    =", skipped)

c:\Users\User\anaconda3\envs\monai\Lib\site-packages\huggingface_hub\file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


1.3.6.1.4.1.9328.50.4.0013_seg.nii.gz:   0%|          | 0.00/371k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


1.3.6.1.4.1.9328.50.4.0014.nii.gz:   0%|          | 0.00/184M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


1.3.6.1.4.1.9328.50.4.0327.nii.gz:   0%|          | 0.00/103M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


1.3.6.1.4.1.9328.50.4.0327_seg.nii.gz:   0%|          | 0.00/262k [00:00<?, ?B/s]

done
downloaded = 305
skipped    = 0


In [9]:
# แสดงอันดับทั้งหมด (รันก่อนเลือก SELECT_RANKS)
print("รายการอันดับทั้งหมด:")
for idx, p in enumerate(pairs, 1):
    print(f"  {idx}. {p['patient_id']}")
print("-" * 40)

รายการอันดับทั้งหมด:
  1. 1.3.6.1.4.1.9328.50.4.0001
  2. 1.3.6.1.4.1.9328.50.4.0002
  3. 1.3.6.1.4.1.9328.50.4.0003
  4. 1.3.6.1.4.1.9328.50.4.0004
  5. 1.3.6.1.4.1.9328.50.4.0005
  6. 1.3.6.1.4.1.9328.50.4.0006
  7. 1.3.6.1.4.1.9328.50.4.0007
  8. 1.3.6.1.4.1.9328.50.4.0008
  9. 1.3.6.1.4.1.9328.50.4.0009
  10. 1.3.6.1.4.1.9328.50.4.0010
  11. 1.3.6.1.4.1.9328.50.4.0011
  12. 1.3.6.1.4.1.9328.50.4.0012
  13. 1.3.6.1.4.1.9328.50.4.0013
  14. 1.3.6.1.4.1.9328.50.4.0014
  15. 1.3.6.1.4.1.9328.50.4.0015
  16. 1.3.6.1.4.1.9328.50.4.0016
  17. 1.3.6.1.4.1.9328.50.4.0017
  18. 1.3.6.1.4.1.9328.50.4.0018
  19. 1.3.6.1.4.1.9328.50.4.0019
  20. 1.3.6.1.4.1.9328.50.4.0020
  21. 1.3.6.1.4.1.9328.50.4.0021
  22. 1.3.6.1.4.1.9328.50.4.0023
  23. 1.3.6.1.4.1.9328.50.4.0024
  24. 1.3.6.1.4.1.9328.50.4.0025
  25. 1.3.6.1.4.1.9328.50.4.0026
  26. 1.3.6.1.4.1.9328.50.4.0027
  27. 1.3.6.1.4.1.9328.50.4.0028
  28. 1.3.6.1.4.1.9328.50.4.0029
  29. 1.3.6.1.4.1.9328.50.4.0030
  30. 1.3.6.1.4.1.9328.50.4.003

In [6]:
# ตรวจสอบไฟล์ที่โหลดมา
print("📁 dummy/images/")
images = sorted(IMAGES_DIR.glob("*.nii.gz"))
for img in images[:10]:  # แสดง 10 ไฟล์แรก
    print(f"   {img.name}")
if len(images) > 10:
    print(f"   ... และอีก {len(images)-10} ไฟล์")

print(f"\n📁 dummy/masks/")
masks = sorted(MASKS_DIR.glob("*.nii.gz"))
for msk in masks[:10]:
    print(f"   {msk.name}")
if len(masks) > 10:
    print(f"   ... และอีก {len(masks)-10} ไฟล์")

print(f"\n📊 Summary: {len(images)} images, {len(masks)} masks")

📁 dummy/images/

📁 dummy/masks/

📊 Summary: 0 images, 0 masks


## ทดสอบดูข้อมูลที่โหลดมา

In [7]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

# โหลดและแสดงตัวอย่าง
if len(images) > 0:
    # โหลดไฟล์แรก
    img_path = images[0]
    mask_path = MASKS_DIR / img_path.name
    
    print(f"Loading: {img_path.name}")
    
    volume = nib.load(str(img_path)).get_fdata()
    mask = nib.load(str(mask_path)).get_fdata()
    
    print(f"Volume shape: {volume.shape}")
    print(f"Mask shape: {mask.shape}")
    print(f"Volume range: [{volume.min():.1f}, {volume.max():.1f}] HU")
    print(f"Unique labels: {np.unique(mask)}")
    
    # แสดง middle slice
    mid_slice = volume.shape[2] // 2
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(volume[:, :, mid_slice].T, cmap='gray', origin='lower')
    axes[0].set_title(f'CT Volume (slice {mid_slice})')
    
    axes[1].imshow(mask[:, :, mid_slice].T, cmap='jet', origin='lower')
    axes[1].set_title('Segmentation Mask')
    
    axes[2].imshow(volume[:, :, mid_slice].T, cmap='gray', origin='lower')
    axes[2].imshow(mask[:, :, mid_slice].T, cmap='jet', alpha=0.4, origin='lower')
    axes[2].set_title('Overlay')
    
    plt.tight_layout()
    plt.show()
else:
    print("No files found. Run the download cell first.")

No files found. Run the download cell first.
